In [4]:
import numpy as np
import pandas as pd

from evidently import Report
from evidently.presets import DataDriftPreset

print("Monitoring imports: OK")

Monitoring imports: OK


In [2]:
import data_prep as dp
import config as cfg

X, y, X_train, X_val, X_test, y_train, y_val, y_test = dp.prepare(cfg.DATA_PATH)

print("X shape:", X.shape)
print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("X_test shape:", X_test.shape)
print("Training positive rate:", round(y_train.mean(), 4))

c:\Users\mithu\Music\MLProjects\hospital-readmission-prediction\data_prep.py:75: DtypeWarning: Columns (0: payer_code) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_path, na_values=["?"])


X shape: (69973, 54)
X_train shape: (48981, 54)
X_val shape: (10496, 54)
X_test shape: (10496, 54)
Training positive rate: 0.0897


In [3]:
numeric_features, categorical_features = dp.get_feature_lists(X)

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("\nNumeric:")
print(numeric_features)
print("\nCategorical:")
print(categorical_features)

Numeric features: 11
Categorical features: 35

Numeric:
['time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses', 'age_midpoint', 'total_prior_visits', 'medication_change_count']

Categorical:
['race', 'gender', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'payer_code', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'diag_1_category', 'diag_2_category', 'diag_3_category', 'medical_specialty_topk']


In [4]:
import inspect
print(inspect.getsource(dp.prepare))

def prepare(data_path):

    df = load_and_prepare_data(data_path)

    X = df.drop(
        columns=["target"]
    ).copy()

    y = df["target"].copy()

    # --------------------------------------------------------
    # 70 / 15 / 15 stratified split
    # --------------------------------------------------------

    X_train, X_temp, y_train, y_temp = train_test_split(
        X,
        y,
        test_size=0.30,
        stratify=y,
        random_state=RANDOM_STATE
    )

    X_val, X_test, y_val, y_test = train_test_split(
        X_temp,
        y_temp,
        test_size=0.50,
        stratify=y_temp,
        random_state=RANDOM_STATE
    )

    return (
        X,
        y,
        X_train,
        X_val,
        X_test,
        y_train,
        y_val,
        y_test
    )



In [1]:
import importlib
import data_prep as dp
import config as cfg

importlib.reload(dp)

X, y, X_train, X_val, X_test, y_train, y_val, y_test = dp.prepare(cfg.DATA_PATH)

print("X shape:", X.shape)
print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("X_test shape:", X_test.shape)
print("Feature count:", X.shape[1])
print("Training positive rate:", round(y_train.mean(), 4))

c:\Users\mithu\Music\MLProjects\hospital-readmission-prediction\data_prep.py:75: DtypeWarning: Columns (0: payer_code) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_path, na_values=["?"])


X shape: (69973, 46)
X_train shape: (48981, 46)
X_val shape: (10496, 46)
X_test shape: (10496, 46)
Feature count: 46
Training positive rate: 0.0897


In [2]:
reference_sample = X_train.sample(
    n=min(5000, len(X_train)),
    random_state=cfg.RANDOM_STATE
).copy()

reference_sample.to_csv("reference_sample.csv", index=False)

print("Reference sample shape:", reference_sample.shape)
print("Saved: reference_sample.csv")

Reference sample shape: (5000, 46)
Saved: reference_sample.csv


In [5]:
current_batch = X_test.sample(
    n=min(5000, len(X_test)),
    random_state=cfg.RANDOM_STATE
).copy()

current_batch["time_in_hospital"] = (
    current_batch["time_in_hospital"] + 2
).clip(lower=1)

current_batch["num_medications"] = (
    current_batch["num_medications"] + 3
)

current_batch["age_midpoint"] = (
    current_batch["age_midpoint"] + 5
).clip(upper=95)

current_batch["diag_1_category"] = np.where(
    np.random.default_rng(cfg.RANDOM_STATE).random(len(current_batch)) < 0.20,
    "Circulatory",
    current_batch["diag_1_category"]
)

print("Current batch shape:", current_batch.shape)
print("\nMean comparison:")
print(
    pd.DataFrame({
        "reference_mean": reference_sample[
            ["time_in_hospital", "num_medications", "age_midpoint"]
        ].mean(),
        "current_mean": current_batch[
            ["time_in_hospital", "num_medications", "age_midpoint"]
        ].mean()
    })
)
print("\nCurrent Circulatory share:",
      round((current_batch["diag_1_category"] == "Circulatory").mean(), 4))

Current batch shape: (5000, 46)

Mean comparison:
                  reference_mean  current_mean
time_in_hospital          4.3242        6.2980
num_medications          15.6200       18.6706
age_midpoint             65.3560       70.2600

Current Circulatory share: 0.4544


In [6]:
current_batch.to_csv("current_batch.csv", index=False)

print("Current batch saved: current_batch.csv")
print("Shape:", current_batch.shape)

Current batch saved: current_batch.csv
Shape: (5000, 46)


In [7]:
drift_report = Report(
    metrics=[
        DataDriftPreset()
    ]
)

drift_report.run(
    reference_data=reference_sample,
    current_data=current_batch
)

print("Evidently feature drift report: generated successfully")

Evidently feature drift report: generated successfully


In [11]:
print([method for method in dir(drift_report) if "html" in method.lower() or "save" in method.lower() or "export" in method.lower()])

[]


In [12]:
print([method for method in dir(drift_report) if not method.startswith("_")])

['include_tests', 'items', 'metadata', 'metrics', 'run', 'set_batch_size', 'set_dataset_id', 'set_model_id', 'set_reference_id', 'tags']


In [13]:
import inspect

print(inspect.signature(drift_report.run))

(current_data: Union[ForwardRef('Dataset'), pandas.DataFrame], reference_data: Union[ForwardRef('Dataset'), pandas.DataFrame, NoneType] = None, additional_data: Optional[Dict[str, Union[ForwardRef('Dataset'), pandas.DataFrame]]] = None, timestamp: Optional[datetime.datetime] = None, metadata: Dict[str, Union[str, Dict[str, str], List[str]]] = None, tags: List[str] = None, name: Optional[str] = None) -> evidently.core.report.Snapshot


In [14]:
snapshot = drift_report.run(
    reference_data=reference_sample,
    current_data=current_batch
)

print(type(snapshot))
print([method for method in dir(snapshot) if not method.startswith("_")])

<class 'evidently.core.report.Snapshot'>
['context', 'dict', 'dump_dict', 'dumps', 'get_html_str', 'get_name', 'json', 'load', 'load_dict', 'load_model', 'loads', 'metric_results', 'render_only_fingerprint', 'report', 'run', 'save_html', 'save_json', 'set_name', 'tests_results', 'to_snapshot_model']


In [15]:
snapshot.save_html("drift_report.html")

print("Saved: drift_report.html")

Saved: drift_report.html


In [16]:
results = snapshot.dict()

print("Top-level result keys:")
print(results.keys())

Top-level result keys:
dict_keys(['metrics', 'tests'])


In [17]:
for metric in results["metrics"]:
    print(metric)

{'id': '15e89f895b482f9b84ba7274ed18a106', 'metric_name': 'DriftedColumnsCount(drift_share=0.5)', 'config': {'type': 'evidently:metric_v2:DriftedColumnsCount', 'drift_share': 0.5}, 'value': {'count': 4.0, 'share': 0.08695652173913043}}
{'id': '9a5ea1f9f373117f2023f2bf018a3fbd', 'metric_name': 'ValueDrift(column=time_in_hospital,method=Wasserstein distance (normed),threshold=0.1)', 'config': {'type': 'evidently:metric_v2:ValueDrift', 'column': 'time_in_hospital', 'method': 'Wasserstein distance (normed)', 'threshold': 0.1}, 'value': np.float64(0.6670415517012063)}
{'id': '6be13de6cf3a22130910a7a4df24114e', 'metric_name': 'ValueDrift(column=num_lab_procedures,method=Wasserstein distance (normed),threshold=0.1)', 'config': {'type': 'evidently:metric_v2:ValueDrift', 'column': 'num_lab_procedures', 'method': 'Wasserstein distance (normed)', 'threshold': 0.1}, 'value': np.float64(0.013159325128666011)}
{'id': '11c9d9b52e494ee03072fcbfb7f26b17', 'metric_name': 'ValueDrift(column=num_medicatio

In [18]:
drift_metric = results["metrics"][0]

print("Drifted features:", int(drift_metric["value"]["count"]))
print("Drift share:", round(drift_metric["value"]["share"], 4))

Drifted features: 4
Drift share: 0.087


In [19]:
drifted_features = [
    metric["metric_name"].split("column=")[1].split(",")[0]
    for metric in results["metrics"][1:]
    if "threshold=0.1" in metric["metric_name"]
    and (
        (metric["config"]["type"].endswith("ValueDrift")
         and (
             ("Wasserstein" in metric["metric_name"] and metric["value"] > 0.1)
             or ("Jensen-Shannon" in metric["metric_name"] and metric["value"] > 0.1)
         ))
    )
]

print("Drifted features:", drifted_features)

Drifted features: ['time_in_hospital', 'num_medications', 'age_midpoint', 'diag_1_category']


In [20]:
drift_summary = pd.DataFrame({
    "metric": [
        "Total features",
        "Drifted features",
        "Drift share",
        "Dataset drift flagged"
    ],
    "value": [
        len(X_train.columns),
        len(drifted_features),
        round(len(drifted_features) / len(X_train.columns), 4),
        len(drifted_features) / len(X_train.columns) > 0.30
    ]
})

display(drift_summary)

,metric,value
0,Total features,46
1,Drifted features,4
2,Drift share,0.087
3,Dataset drift flagged,False


In [21]:
import mlflow

model_path = "mlruns/1/models/m-d91399435d7443b1ab3a508644f43793/artifacts"

model = mlflow.sklearn.load_model(model_path)

print("Model loaded:", type(model))

Model loaded: <class 'sklearn.pipeline.Pipeline'>


In [22]:
reference_scores = model.predict_proba(reference_sample)[:, 1]
current_scores = model.predict_proba(current_batch)[:, 1]

print("Reference scores:", len(reference_scores))
print("Current scores:", len(current_scores))
print("Reference mean:", round(reference_scores.mean(), 4))
print("Current mean:", round(current_scores.mean(), 4))

Reference scores: 5000
Current scores: 5000
Reference mean: 0.4402
Current mean: 0.4605


In [23]:
def calculate_psi(reference, current, bins=10):
    edges = np.quantile(reference, np.linspace(0, 1, bins + 1))
    edges = np.unique(edges)

    reference_counts = np.histogram(reference, bins=edges)[0]
    current_counts = np.histogram(current, bins=edges)[0]

    reference_pct = reference_counts / len(reference)
    current_pct = current_counts / len(current)

    epsilon = 1e-6
    reference_pct = np.clip(reference_pct, epsilon, None)
    current_pct = np.clip(current_pct, epsilon, None)

    return np.sum(
        (current_pct - reference_pct)
        * np.log(current_pct / reference_pct)
    )

prediction_psi = calculate_psi(
    reference_scores,
    current_scores
)

print("Prediction PSI:", round(prediction_psi, 4))

Prediction PSI: 0.0262


In [24]:
import json

drift_summary = {
    "prediction_psi": round(float(prediction_psi), 4),
    "psi_threshold": 0.20,
    "prediction_drift": bool(prediction_psi > 0.20),
    "drifted_features": drifted_features,
    "drifted_feature_share": round(len(drifted_features) / len(X_train.columns), 4),
    "drift_share_threshold": 0.30,
    "dataset_drift": bool(len(drifted_features) / len(X_train.columns) > 0.30)
}

with open("drift_summary.json", "w") as f:
    json.dump(drift_summary, f, indent=2)

print("Saved: drift_summary.json")

Saved: drift_summary.json


In [25]:
print("=== MONITORING SUMMARY ===")
print("Reference sample:", reference_sample.shape)
print("Current batch:", current_batch.shape)
print("Drifted features:", len(drifted_features))
print("Drift share:", round(len(drifted_features) / len(X_train.columns), 4))
print("Dataset drift:", len(drifted_features) / len(X_train.columns) > 0.30)
print("Prediction PSI:", round(prediction_psi, 4))
print("Prediction drift:", prediction_psi > 0.20)

=== MONITORING SUMMARY ===
Reference sample: (5000, 46)
Current batch: (5000, 46)
Drifted features: 4
Drift share: 0.087
Dataset drift: False
Prediction PSI: 0.0262
Prediction drift: False


In [26]:
from retrain import should_retrain

trigger_result = should_retrain(
    prediction_psi=prediction_psi,
    drifted_feature_share=len(drifted_features) / len(X_train.columns),
    dataset_drift=False
)

print(trigger_result)

{'prediction_psi': np.float64(0.026230778862146385), 'psi_threshold': 0.2, 'psi_trigger': np.False_, 'drifted_feature_share': 0.08695652173913043, 'drift_share_threshold': 0.3, 'feature_drift_trigger': False, 'dataset_drift': False, 'dataset_drift_trigger': False, 'retrain': False}


In [27]:
test_trigger = should_retrain(
    prediction_psi=0.25,
    drifted_feature_share=0.087,
    dataset_drift=False
)

print(test_trigger)

{'prediction_psi': 0.25, 'psi_threshold': 0.2, 'psi_trigger': True, 'drifted_feature_share': 0.087, 'drift_share_threshold': 0.3, 'feature_drift_trigger': False, 'dataset_drift': False, 'dataset_drift_trigger': False, 'retrain': True}


In [28]:
test_trigger = should_retrain(
    prediction_psi=0.0262,
    drifted_feature_share=0.35,
    dataset_drift=False
)

print(test_trigger)

{'prediction_psi': 0.0262, 'psi_threshold': 0.2, 'psi_trigger': False, 'drifted_feature_share': 0.35, 'drift_share_threshold': 0.3, 'feature_drift_trigger': True, 'dataset_drift': False, 'dataset_drift_trigger': False, 'retrain': True}


In [ ]:
test_trigger = should_retrain(
    prediction_psi=0.0262,
    drifted_feature_share=0.087,
    dataset_drift=True
)

print(test_trigger)

{'prediction_psi': 0.0262, 'psi_threshold': 0.2, 'psi_trigger': False, 'drifted_feature_share': 0.087, 'drift_share_threshold': 0.3, 'feature_drift_trigger': False, 'dataset_drift': True, 'dataset_drift_trigger': True, 'retrain': True}
